In [41]:
import pandas as pd

In [42]:
pd.read_csv('Data/GTFS_Moventis/routes.txt')

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,161,3,C-13,SERVEI URBÀ DE VILASSAR MAR,NaN,3,NaN,09950F,FFFFE9
1,266,3,808,TEIÀ-EL MASNOU-ALELLA-BARCELONA,NaN,3,NaN,DB000E,010619
2,335,3,806,CABRILS-VILASSAR DE DALT-BARCELONA,NaN,3,NaN,FF0111,FFFFB5
3,340,3,C-19,SERVEI URBÀ DEL MASNOU,NaN,3,NaN,540074,FFFFAA
4,385,3,C-18,SERVEI URBÀ DE PREMIA DE MAR,NaN,3,NaN,E70008,02032A
5,47,3,e11.1,MATARO (CENTRE) - BARCELONA (PER AUTOPISTA),NaN,3,NaN,0BBC90,FFFFFF
6,48,3,e11.2,MATARO (NORD) - BARCELONA (PER AUTOPISTA),NaN,3,NaN,0BBC90,FFFFFF
7,49,3,805,803 / 804 VILASSAR D. - PREMIÀ M. - BCN,NaN,3,NaN,EE1B00,FFFFFF
8,51,3,865,MATARÓ - BELLATERRA (UAB),NaN,3,NaN,FF860A,FFFFE2
9,53,3,C-10,MATARO - BARCELONA (PER N-II),NaN,3,NaN,9AFF1F,011013


In [43]:
routes = pd.read_csv('Data/GTFS_Moventis/routes.txt')
routes = routes[routes['route_short_name'] == 'e11.1'] # Only bus routes    
route_id = routes['route_id'].values[0]

In [44]:
trips = pd.read_csv('Data/GTFS_Moventis/trips.txt')
trips

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
0,161,16100242-DISSABTES DIUMENGE I FESTIUS ESTIU,161000020341,REC. CURT,NaN,0,NaN,0161_0002,1
1,161,16100242-DISSABTES DIUMENGE I FESTIUS ESTIU,161000020357,REC. CURT,NaN,0,NaN,0161_0002,1
2,161,16100242-FEINERS,161000020251,REC. CURT,NaN,0,NaN,0161_0002,1
3,161,16100242-FEINERS,161000020252,REC. CURT,NaN,0,NaN,0161_0002,1
4,161,16100242-FEINERS,161000020253,REC. CURT,NaN,0,NaN,0161_0002,1
...,...,...,...,...,...,...,...,...,...
1733,715,71502508-FEINERS,715000010018,TEIA-MASNOU-TEIA,NaN,0,NaN,0715_0001,2
1734,715,71502508-FEINERS,715000010019,TEIA-MASNOU-TEIA,NaN,0,NaN,0715_0001,2
1735,715,71502508-FEINERS,715000010020,TEIA-MASNOU-TEIA,NaN,0,NaN,0715_0001,2
1736,715,71502508-FEINERS,715000010021,TEIA-MASNOU-TEIA,NaN,0,NaN,0715_0001,2


In [59]:
trips = pd.read_csv('Data/GTFS_Moventis/trips.txt')
trips = trips[trips['route_id'] == route_id]
trips = trips[trips['trip_headsign'] == 'MAT(CENTRE)-BCN']
trip_ids = trips['trip_id'].to_list()

In [46]:
stop_times = pd.read_csv('Data/GTFS_Moventis/stop_times.txt')
stop_times = stop_times[stop_times['trip_id'].isin(trip_ids)][['trip_id','arrival_time','stop_id','stop_sequence']]
stop_times = stop_times[stop_times['arrival_time'].str.slice(0,2).astype(int) >= 6]
stop_times = stop_times[stop_times['arrival_time'].str.slice(0,2).astype(int) < 11]

stops = pd.read_csv('Data/GTFS_Moventis/stops.txt')[['stop_id','stop_name']]
stop_times = stop_times.merge(stops, on='stop_id', how='left')
trip_lengths = stop_times.groupby('trip_id').size().reset_index(name='length')
trip_lengths = trip_lengths[trip_lengths['length'] == 8]
stop_times = stop_times[stop_times['trip_id'].isin(trip_lengths['trip_id'])]
stop_times


,trip_id,arrival_time,stop_id,stop_sequence,stop_name
24,47000011413,09:50:00,7077,1,PL. DE LES TERESES
25,47000011413,09:51:00,13521,2,C. DE SANT ISIDOR
26,47000011413,09:53:00,21749,3,CAMÍ DE LA GEGANTA/PL. GRANOLLERS (POLICIA LOCAL)
27,47000011413,09:56:00,7048,4,RDA. DE LA REPÚBLICA / C. DE MIQUEL BIADA
28,47000011413,09:58:00,7159,5,CAMÍ RAL/C. PIZARRO (PL. DR. FLEMING)
...,...,...,...,...,...
195,47000011392,06:11:00,7048,4,RDA. DE LA REPÚBLICA / C. DE MIQUEL BIADA
196,47000011392,06:13:00,7159,5,CAMÍ RAL/C. PIZARRO (PL. DR. FLEMING)
197,47000011392,06:15:00,7160,6,CAMÍ RAL (PORTA LAIETANA-TECNOCAMPUS)
198,47000011392,06:50:00,7506,7,PL.TETUAN (GVCC /NÀPOLS)


In [34]:
frequency = stop_times['trip_id'].nunique() / 5
frequency

4.4

calcular el que es tarda arribar a la city gate. primera parada -> tetuan - 3 min

In [ ]:
stop_times_end_start = stop_times[stop_times['stop_sequence'].isin([1,7])]
stop_times_pivot = stop_times_end_start.pivot(index='trip_id', columns='stop_name', values='arrival_time').reset_index()
stop_times_pivot['travel_time'] = (
    pd.to_timedelta(stop_times_pivot['PL.TETUAN (GVCC /NÀPOLS)'])
    - pd.to_timedelta(stop_times_pivot['PL. DE LES TERESES'])
).dt.total_seconds() / 60
stop_times_pivot

stop_name,trip_id,PL. DE LES TERESES,PL.TETUAN (GVCC /NÀPOLS),travel_time
0,47000011392,06:05:00,06:50:00,45.0
1,47000011393,06:15:00,07:00:00,45.0
2,47000011394,06:25:00,07:10:00,45.0
3,47000011395,06:35:00,07:20:00,45.0
4,47000011396,06:45:00,07:30:00,45.0
5,47000011397,06:55:00,07:40:00,45.0
6,47000011398,07:05:00,07:50:00,45.0
7,47000011399,07:15:00,08:00:00,45.0
8,47000011400,07:25:00,08:10:00,45.0
9,47000011401,07:35:00,08:20:00,45.0


In [36]:
stop_times_pivot['travel_time'].mean()

np.float64(45.22727272727273)

In [71]:
trips = pd.read_csv('Data/GTFS_Moventis/trips.txt')
trips = trips[trips['route_id'] == route_id]
trips = trips[trips['trip_headsign'].isin(['MAT(CENTRE)-BCN','BCN - MAT(CENTRE)'])][['trip_id','trip_headsign']]
trip_ids = trips['trip_id'].to_list()
print(f"Number of trips: {len(trip_ids)}")

Number of trips: 188


In [80]:
stop_times = pd.read_csv('Data/GTFS_Moventis/stop_times.txt')
stop_times = stop_times[stop_times['trip_id'].isin(trip_ids)][['trip_id','arrival_time','stop_id','stop_sequence']]
stop_times = stop_times[stop_times['arrival_time'].str.slice(0,2).astype(int) >= 6]
stop_times = stop_times[stop_times['arrival_time'].str.slice(0,2).astype(int) < 11]
stop_times = stop_times.merge(trips, on='trip_id', how='left')
stop_times

stops = pd.read_csv('Data/GTFS_Moventis/stops.txt')[['stop_id','stop_name']]
stop_times = stop_times.merge(stops, on='stop_id', how='left')
trip_lengths = stop_times.groupby(['trip_id', 'trip_headsign']).size().reset_index(name='length')
# select BCN - MAT(CENTRE) and lenght 7 or MAT(CENTRE)-BCN and lenght 8
trip_lengths = trip_lengths[((trip_lengths['trip_headsign'] == 'BCN - MAT(CENTRE)') & (trip_lengths['length'] == 7)) | ((trip_lengths['trip_headsign'] == 'MAT(CENTRE)-BCN') & (trip_lengths['length'] == 8))]
trip_lengths = trip_lengths[trip_lengths['length'].isin([8,7])]
stop_times = stop_times[stop_times['trip_id'].isin(trip_lengths['trip_id'])]
stop_times

,trip_id,arrival_time,stop_id,stop_sequence,trip_headsign,stop_name
32,47000011413,09:50:00,7077,1,MAT(CENTRE)-BCN,PL. DE LES TERESES
33,47000011413,09:51:00,13521,2,MAT(CENTRE)-BCN,C. DE SANT ISIDOR
34,47000011413,09:53:00,21749,3,MAT(CENTRE)-BCN,CAMÍ DE LA GEGANTA/PL. GRANOLLERS (POLICIA LOCAL)
35,47000011413,09:56:00,7048,4,MAT(CENTRE)-BCN,RDA. DE LA REPÚBLICA / C. DE MIQUEL BIADA
36,47000011413,09:58:00,7159,5,MAT(CENTRE)-BCN,CAMÍ RAL/C. PIZARRO (PL. DR. FLEMING)
...,...,...,...,...,...,...
373,47000020500,07:25:00,7216,3,BCN - MAT(CENTRE),C. DE LA BOBINADORA (POL. IND. LES HORTES)
374,47000020500,07:27:00,7217,4,BCN - MAT(CENTRE),CAMÍ RAL (PORTA LAIETANA-TECNOCAMPUS)
375,47000020500,07:29:00,7218,5,BCN - MAT(CENTRE),RDA. DE LA REPÚBLICA / CAMÍ RAL
376,47000020500,07:32:00,7219,6,BCN - MAT(CENTRE),RDA. DE LA REPÚBLICA / C. DE MIQUEL BIADA


In [85]:
import pandas as pd
TERM_STOPS = [
    "PL. DE LES TERESES",          
    "RONDA DE LA UNIVERSITAT 25"  # last stop of the line
]

term_df = stop_times[stop_times["stop_name"].isin(TERM_STOPS)].copy()

def hhmmss_to_seconds(t: str) -> int:
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

term_df["arr_seconds"] = term_df["arrival_time"].apply(hhmmss_to_seconds)
term_df = term_df.sort_values(["stop_name", "arr_seconds"])

term_df["break_sec"] = term_df.groupby("stop_name")["arr_seconds"].diff()

term_df = term_df.dropna(subset=["break_sec"])

avg_break_by_stop = term_df.groupby("stop_name")["break_sec"].mean()
overall_avg_sec = term_df["break_sec"].mean()
overall_avg_sec / 60

np.float64(5.833333333333333)